In [ ]:
from google.colab import drive

# Google Drive mount karo
drive.mount('/content/drive')

print(" Drive mounted successfully!")

Mounted at /content/drive
 Drive mounted successfully!


In [ ]:
import tensorflow as tf

device_name = tf.test.gpu_device_name()
if device_name:
    print(f" GPU connected: {device_name}")
else:
    print(" No GPU found. Currently using CPU.")


 GPU connected: /device:GPU:0


In [ ]:
import shutil

# Drive se dataset ka path
drive_dataset_path = "/content/drive/MyDrive/Pneumonia_90_10_Splitted"

# Colab local storage path
local_dataset_path = "/content/Pneumonia_90_10_Splitted"

# Agar already local me hai to pehle delete kar do (fresh copy ke liye)
shutil.rmtree(local_dataset_path, ignore_errors=True)

# Copy from Drive to local
shutil.copytree(drive_dataset_path, local_dataset_path)

print(" Dataset copied to local Colab storage:", local_dataset_path)


 Dataset copied to local Colab storage: /content/Pneumonia_90_10_Splitted


In [ ]:
# ============================================
# Pneumonia Detection with DenseNet169 (Final + Class Weights)
# ============================================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.densenet import DenseNet169, preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
from google.colab import drive

# =========================
# Step 1: Mount Google Drive
# =========================
drive.mount('/content/drive')

# Paths
BASE_DIR = "/content/drive/MyDrive/pneumonia_detection_system"
MODEL_DIR = os.path.join(BASE_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

# Updated model path (DenseNet169)
MODEL_PATH = os.path.join(MODEL_DIR, "densenet169_pneumonia_model.keras")

# Dataset path (train part only from 90-10 split)
DATASET_DIR = "/content/Pneumonia_90_10_Splitted/train"

# =========================
# Step 2: Data Preprocessing
# =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    DATASET_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

# =========================
# Step 2.1: Compute Class Weights
# =========================
labels = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights_array))
print("Computed class weights:", class_weights)

# =========================
# Step 3: Build DenseNet169 Model
# =========================
base_model = DenseNet169(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze base for initial training

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# =========================
# Step 4: Training Phase 1 (Frozen Base)
# =========================
checkpoint = ModelCheckpoint(MODEL_PATH, save_best_only=True)

history1 = model.fit(
    train_generator,
    epochs=20,  # Half of total
    callbacks=[checkpoint],
    class_weight=class_weights   # <<<<<< added
)

# =========================
# Step 5: Fine-Tuning Phase 2 (Unfreeze Last Layers)
# =========================
base_model.trainable = True
for layer in base_model.layers[:-50]:  # Freeze all except last 50 layers
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history2 = model.fit(
    train_generator,
    epochs=20,  # Second half (total = 40)
    callbacks=[checkpoint],
    class_weight=class_weights   # <<<<<< added
)

# =========================
# Step 6: Save Final Model
# =========================
model.save(MODEL_PATH)
print(f" Model saved at {MODEL_PATH}")

# =========================
# Step 7: Print last Conv layer for Grad-CAM
# =========================
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        print(" Last Conv Layer for Grad-CAM:", layer.name)
        break


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 13500 images belonging to 3 classes.
Computed class weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0)}
51877672/51877672 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 0s 441ms/step - accuracy: 0.6290 - loss: 0.8251

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/model_checkpoint.py:302: UserWarning: Can save best model only with val_loss available.
  if self._should_save_model(epoch, batch, logs, filepath):


422/422 ━━━━━━━━━━━━━━━━━━━━ 229s 445ms/step - accuracy: 0.6291 - loss: 0.8248
Epoch 2/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 168s 399ms/step - accuracy: 0.7622 - loss: 0.5577
Epoch 3/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 202s 399ms/step - accuracy: 0.7805 - loss: 0.5117
Epoch 4/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 202s 398ms/step - accuracy: 0.7868 - loss: 0.5048
Epoch 5/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 202s 399ms/step - accuracy: 0.7964 - loss: 0.4818
Epoch 6/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 169s 400ms/step - accuracy: 0.7967 - loss: 0.4751
Epoch 7/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 201s 398ms/step - accuracy: 0.8036 - loss: 0.4683
Epoch 8/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 203s 402ms/step - accuracy: 0.8135 - loss: 0.4499
Epoch 9/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 198s 393ms/step - accuracy: 0.8122 - loss: 0.4483
Epoch 10/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 165s 392ms/step - accuracy: 0.8110 - loss: 0.4460
Epoch 11/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 166s 393ms/step - accuracy: 0.8173 - loss: 0.4386
Epoch 12/20
422/422

In [ ]:
# ============================================
# Step: Confusion Matrix on Test Dataset (DenseNet169)
# ============================================
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model

# -------------------------------
# Paths
# -------------------------------
BASE_DIR = "/content/drive/MyDrive/pneumonia_detection_system"
MODEL_PATH = os.path.join(BASE_DIR, "model", "densenet169_pneumonia_model.keras")  # DenseNet169 model
TEST_DIR = "/content/Pneumonia_90_10_Splitted/test"   # <-- test part
EVAL_DIR = os.path.join(BASE_DIR, "evaluation")
os.makedirs(EVAL_DIR, exist_ok=True)

# -------------------------------
# Load trained DenseNet169 model
# -------------------------------
model = load_model(MODEL_PATH)
print(" Model loaded from:", MODEL_PATH)

# -------------------------------
# Image generator for test set
# -------------------------------
IMG_SIZE = (224, 224)
CLASS_NAMES = ["Bacterial Pneumonia", "Normal", "Viral Pneumonia"]

datagen = ImageDataGenerator(rescale=1.0/255.0)
test_generator = datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

# -------------------------------
# True labels
# -------------------------------
y_true = test_generator.classes

# -------------------------------
# Model predictions
# -------------------------------
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# -------------------------------
# Confusion Matrix
# -------------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix on Test Set (DenseNet169)")

# Save confusion matrix
cm_path = os.path.join(EVAL_DIR, "DenseNet169_confusion_matrix.png")
plt.savefig(cm_path, dpi=300)
plt.close()

print(f" Confusion matrix generated and saved at: {cm_path}")

# -------------------------------
# Classification Report
# -------------------------------
print("\n Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


 Model loaded from: /content/drive/MyDrive/pneumonia_detection_system/model/densenet169_pneumonia_model.keras
Found 1500 images belonging to 3 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


47/47 ━━━━━━━━━━━━━━━━━━━━ 41s 502ms/step
 Confusion matrix generated and saved at: /content/drive/MyDrive/pneumonia_detection_system/evaluation/DenseNet169_confusion_matrix.png

 Classification Report:
                     precision    recall  f1-score   support

Bacterial Pneumonia       0.71      0.73      0.72       500
             Normal       0.98      0.68      0.80       500
    Viral Pneumonia       0.59      0.75      0.66       500

           accuracy                           0.72      1500
          macro avg       0.76      0.72      0.73      1500
       weighted avg       0.76      0.72      0.73      1500



In [4]:
# ============================================
# PNEUMOSCAN — Multi-Class (3-Class) Evaluator with UI Button
# Uses your saved DenseNet169 model and sampled test set (100 images per class)
# ============================================

# ---- Imports ----
import os, numpy as np, matplotlib.pyplot as plt
import random, shutil, glob
from itertools import product
from IPython.display import display, clear_output
import ipywidgets as widgets

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_fscore_support
)

# ---- 1) Mount Drive ----
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount note:", e)

# ---- 2) Fixed Paths ----
BASE_DIR = "/content/drive/MyDrive/pneumonia_detection_system"
MODEL_PATH = os.path.join(BASE_DIR, "model", "densenet169_pneumonia_model.keras")
SOURCE_TEST_DIR = "/content/drive/MyDrive/Pneumonia_90_10_Splitted/test"
SAMPLED_TEST_DIR = "/content/drive/MyDrive/pneumonia_detection_system/evaluation/sample_test_300"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SAMPLES_PER_CLASS = 100
CLASS_NAMES = ['Bacterial Pneumonia', 'Viral Pneumonia', 'Normal']

# ---- UI Elements ----
hdr = widgets.HTML("<h3> Pneumonia Detection System — Multi-Class Model Evaluator (3 Classes)</h3>")
status_box = widgets.HTML("<b>Status:</b> Initializing...")
evaluate_btn = widgets.Button(description="Evaluate", button_style='primary', tooltip="Run evaluation on sampled test set")
evaluate_btn.layout.width = "200px"
box = widgets.VBox([hdr, status_box, evaluate_btn])
display(box)

# Globals to hold objects across button click
_model = None
_test_gen = None
_class_indices = None

# ---- 3) Helper: Create Sampled Test Folder ----
def create_sampled_test_folder(src_test_dir, dst_sample_dir, class_names, samples_per_class=100, seed=42):
    """Create a test folder with exactly samples_per_class images from each class"""
    random.seed(seed)

    # Remove existing sampled folder if exists
    if os.path.exists(dst_sample_dir):
        shutil.rmtree(dst_sample_dir)

    # Create fresh sampled folder
    os.makedirs(dst_sample_dir, exist_ok=True)

    messages = []
    total_sampled = 0

    for cls in class_names:
        src_cls_dir = os.path.join(src_test_dir, cls)
        dst_cls_dir = os.path.join(dst_sample_dir, cls)
        os.makedirs(dst_cls_dir, exist_ok=True)

        if not os.path.isdir(src_cls_dir):
            messages.append(f"Source class folder missing: {src_cls_dir}")
            continue

        # Get all images from source class folder
        patterns = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff"]
        files = []
        for pattern in patterns:
            files.extend(glob.glob(os.path.join(src_cls_dir, pattern)))

        n_available = len(files)
        if n_available == 0:
            messages.append(f"No images found in {src_cls_dir}")
            continue

        # Sample images
        if n_available <= samples_per_class:
            chosen = files  # Take all if not enough
            messages.append(f"{cls}: Only {n_available} available — taking all.")
        else:
            chosen = random.sample(files, samples_per_class)
            messages.append(f"{cls}: Sampled {samples_per_class} of {n_available} images.")

        # Copy to destination
        for fpath in chosen:
            shutil.copy2(fpath, os.path.join(dst_cls_dir, os.path.basename(fpath)))

        total_sampled += len(chosen)

    messages.append(f"Total: {total_sampled} images sampled.")
    return messages

# ---- 4) Create Sampled Dataset ----
status_box.value = "<b>Status:</b> Creating sampled test dataset (100 images per class)..."
messages = create_sampled_test_folder(
    SOURCE_TEST_DIR,
    SAMPLED_TEST_DIR,
    CLASS_NAMES,
    SAMPLES_PER_CLASS
)
status_box.value = "<b>Status:</b> " + " | ".join(messages)

# ---- 5) Load Model ----
if not os.path.exists(MODEL_PATH):
    status_box.value = f"<b>Status:</b> Model not found at: <code>{MODEL_PATH}</code>"
    raise SystemExit

status_box.value = "<b>Status:</b> Loading model…"
_model = load_model(MODEL_PATH)
status_box.value = "<b>Status:</b> Model loaded."

# ---- 6) Load Sampled Test Dataset ----
if not os.path.isdir(SAMPLED_TEST_DIR):
    status_box.value = f"<b>Status:</b> Sampled test folder not found at: <code>{SAMPLED_TEST_DIR}</code>"
    raise SystemExit

status_box.value = "<b>Status:</b> Loading sampled test set…"

# Use rescale=1./255 for DenseNet169
test_datagen = ImageDataGenerator(rescale=1./255)
_test_gen = test_datagen.flow_from_directory(
    SAMPLED_TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

_class_indices = _test_gen.class_indices
status_box.value = (f"<b>Status:</b> Test set ready "
                    f"(Classes: {_class_indices}, Samples: {len(_test_gen.filenames)}). "
                    f"<br> Click <b>Evaluate</b> to compute metrics.")

# ---- 7) Helper: Pretty Confusion Matrix Plot (Stage-2 style) ----
def plot_confusion_matrix(cm, class_names):
    fig = plt.figure(figsize=(6, 5), dpi=120)
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2.0 if cm.size else 0.5
    for i, j in product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    return fig

# ---- 8) Evaluation on Button Click (Stage-2 style for 3 classes) ----
out_area = widgets.Output()
display(out_area)

def on_evaluate_clicked(b):
    with out_area:
        clear_output(wait=True)
        if _model is None or _test_gen is None:
            print("Model/Test generator not initialized.")
            return

        print(" Running evaluation …")

        # a) Evaluate() for Loss/Accuracy (EXACTLY Stage-2 style)
        loss, acc = _model.evaluate(_test_gen, verbose=0)
        print(f"\n=== Overall Metrics ===")
        print(f"Test Loss     : {loss:.5f}")
        print(f"Test Accuracy : {acc*100:.2f}%")

        # b) Predictions & Confusion Matrix (EXACTLY Stage-2 style)
        y_true = _test_gen.classes
        y_prob = _model.predict(_test_gen, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        # Get class indices
        bacterial_idx = _class_indices.get('Bacterial Pneumonia')
        viral_idx = _class_indices.get('Viral Pneumonia')
        normal_idx = _class_indices.get('Normal')

        # Get class names in correct order
        idx_to_name = {v: k for k, v in _class_indices.items()}
        ordered_names = [idx_to_name[i] for i in range(len(idx_to_name))]

        # Confusion Matrix for 3 classes
        cm = confusion_matrix(y_true, y_pred)

        # c) Per-Class Metrics (EXACTLY Stage-2 style format)
        prfs = precision_recall_fscore_support(
            y_true, y_pred, labels=list(range(len(ordered_names))), zero_division=0
        )
        precision_per_class = prfs[0]
        recall_per_class = prfs[1]
        f1_per_class = prfs[2]
        support_per_class = prfs[3]

        # Macro averages (EXACTLY Stage-2 style)
        precision_macro = prfs[0].mean()
        recall_macro = prfs[1].mean()
        f1_macro = prfs[2].mean()

        # d) AUC-ROC for each class (one-vs-rest) - Stage-2 style
        auc_scores = []
        for class_idx in range(len(ordered_names)):
            y_true_binary = (y_true == class_idx).astype(int)
            class_scores = y_prob[:, class_idx]
            try:
                auc_class = roc_auc_score(y_true_binary, class_scores)
            except ValueError:
                auc_class = float('nan')
            auc_scores.append(auc_class)

        # e) Classification Report (EXACTLY Stage-2 style format)
        print("\n=== Classification Report (per class) ===")
        print(classification_report(
            y_true, y_pred,
            target_names=ordered_names,
            digits=4
        ))

        # f) Confusion Matrix (counts) - Stage-2 style
        print("\n=== Confusion Matrix (counts) ===")
        print(cm)

        # Display confusion matrix values with labels
        print("\nConfusion Matrix Breakdown:")
        for i, true_class in enumerate(ordered_names):
            for j, pred_class in enumerate(ordered_names):
                print(f"  {true_class} → {pred_class}: {cm[i, j]}")

        # g) Key Metrics Summary (Per Class) - Stage-2 style
        print("\n=== Key Metrics (Per Class) ===")
        for i, class_name in enumerate(ordered_names):
            print(f"\n{class_name}:")
            print(f"  Precision : {precision_per_class[i]:.4f}")
            print(f"  Recall    : {recall_per_class[i]:.4f}")
            print(f"  F1-Score  : {f1_per_class[i]:.4f}")
            print(f"  AUC-ROC   : {auc_scores[i]:.4f}")
            print(f"  Support   : {support_per_class[i]}")

        print("\n=== Macro Averages (All 3 Classes) ===")
        print(f"Precision (macro): {precision_macro:.4f}")
        print(f"Recall (macro)   : {recall_macro:.4f}")
        print(f"F1 (macro)       : {f1_macro:.4f}")

        # Calculate Weighted Averages
        weighted_precision = np.average(precision_per_class, weights=support_per_class)
        weighted_recall = np.average(recall_per_class, weights=support_per_class)
        weighted_f1 = np.average(f1_per_class, weights=support_per_class)

        print("\n=== Weighted Averages ===")
        print(f"Precision (weighted): {weighted_precision:.4f}")
        print(f"Recall (weighted)   : {weighted_recall:.4f}")
        print(f"F1 (weighted)       : {weighted_f1:.4f}")

        # h) Visuals — Confusion Matrix Heatmap (Stage-2 style)
        fig_cm = plot_confusion_matrix(cm, ordered_names)
        plt.show(fig_cm)

        # i) ROC Curves for each class (Stage-2 style, but for 3 classes)
        print("\n=== ROC Curves (One-vs-Rest) ===")
        fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=120)

        for idx, class_name in enumerate(ordered_names):
            y_true_binary = (y_true == idx).astype(int)
            class_scores = y_prob[:, idx]

            try:
                fpr, tpr, _ = roc_curve(y_true_binary, class_scores)
                auc_score = auc_scores[idx]

                axes[idx].plot(fpr, tpr, label=f'ROC curve (AUC = {auc_score:.4f})', linewidth=2)
                axes[idx].plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
                axes[idx].set_xlim([0.0, 1.0])
                axes[idx].set_ylim([0.0, 1.05])
                axes[idx].set_xlabel('False Positive Rate')
                axes[idx].set_ylabel('True Positive Rate')
                axes[idx].set_title(f'ROC Curve — {class_name}')
                axes[idx].legend(loc="lower right")
                axes[idx].grid(True, alpha=0.3)
            except Exception as e:
                axes[idx].text(0.5, 0.5, f"ROC unavailable\n{e}",
                              ha='center', va='center', transform=axes[idx].transAxes)

        plt.tight_layout()
        plt.show()

        # j) Additional Statistics
        print("\n=== Dataset Statistics ===")
        print(f"Total Samples Evaluated: {len(y_true)}")
        print(f"Samples per Class: {SAMPLES_PER_CLASS}")
        print(f"Classes Evaluated: {', '.join(ordered_names)}")

        # Calculate overall correct predictions
        correct_predictions = (y_true == y_pred).sum()
        overall_accuracy = correct_predictions / len(y_true)
        print(f"\nCorrect Predictions: {correct_predictions}/{len(y_true)}")
        print(f"Overall Match with evaluate(): {overall_accuracy*100:.2f}% vs {acc*100:.2f}%")

        print("\n Evaluation complete.")

# Only enable button after dataset load (EXACTLY Stage-2 style)
def enable_button_after_load():
    evaluate_btn.disabled = False
    evaluate_btn.button_style = 'success'

evaluate_btn.on_click(on_evaluate_clicked)
enable_button_after_load()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Found 300 images belonging to 3 classes.


Output()